# GEO Dataset Processing for cellxGene

This notebook downloads, processes, and exports three pancreatic scRNA-seq datasets from GEO into
AnnData `.h5ad` format suitable for cellxGene upload and downstream analysis.

| GSE | Description | Format |
|-----|------------|--------|
| GSE154778 | Primary vs metastatic PDAC | CSV DGE matrix + TAR of 10x per-sample |
| GSE162708 | Metastatic pancreatic neuroendocrine tumor (24,544 cells) | Standard 10x (barcodes/features/matrix) |
| GSE165399 | Normal pancreas, IPMN, adenosquamous carcinoma | TAR of TXT DGE matrices |

In [ ]:
# Cell 0: Setup & Imports
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scipy.sparse as sp
import os
import tarfile
import glob
import tempfile
import warnings

%matplotlib inline
warnings.filterwarnings('ignore', category=FutureWarning)
sc.settings.verbosity = 2  # warnings and info

DRIVE_BASE = "/Users/shash/Library/CloudStorage/GoogleDrive-shashvatsmehta@gmail.com/.shortcut-targets-by-id/1m6XoSPhIWcxsfR8TAwcwAbQMXWcFQw6l/ShashvatHighSchoolCollege/2025ResearchExternship/Datasets"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"scanpy version: {sc.__version__}")
print(f"anndata version: {ad.__version__}")
print(f"Drive base: {DRIVE_BASE}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Cell 1: Verify files on Google Drive
#
# No downloads needed — files are accessed directly via Google Drive for Desktop.

expected_files = {
    "GSE154778/GSE154778_dgeMtx.csv.gz": 29,
    "GSE162708/GSE162708_barcodes.tsv.gz": 0.1,
    "GSE162708/GSE162708_features.tsv.gz": 0.2,
    "GSE162708/GSE162708_matrix.mtx.gz": 150,
    "GSE165399/GSE165399_RAW.tar": 18,
}

print("Checking expected files on Google Drive...\n")
all_found = True
for relpath, expected_mb in expected_files.items():
    fpath = os.path.join(DRIVE_BASE, relpath)
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  OK  {relpath} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING  {relpath}")
        all_found = False

# Also check for pre-extracted GSE165399_RAW directory
raw_dir = os.path.join(DRIVE_BASE, "GSE165399", "GSE165399_RAW")
if os.path.isdir(raw_dir):
    n_files = len(os.listdir(raw_dir))
    print(f"  OK  GSE165399/GSE165399_RAW/ ({n_files} files)")
else:
    print(f"  INFO  GSE165399/GSE165399_RAW/ not found (will extract from TAR)")

if all_found:
    print("\nAll expected files found on Google Drive.")
else:
    print("\nWARNING: Some files are missing. Check the Drive path.")

In [ ]:
# Cell 2: Shared Preprocessing Function
#
# Standard scanpy QC + preprocessing pipeline matching the project conventions
# (see CLAUDE.md and DendrogramPlot.py).

def preprocess_adata(adata, min_genes=200, min_cells=3, mt_pct_threshold=20,
                     n_top_genes=2000, n_pcs=30, leiden_resolution=0.5):
    """Run standard scanpy QC and preprocessing pipeline.
    
    Steps:
      1. Make names unique
      2. Filter cells/genes
      3. Mitochondrial QC
      4. Normalize + log1p
      5. Store raw
      6. HVG selection
      7. Scale + PCA + neighbors + UMAP + leiden
    
    Returns the processed AnnData (modifies in place and returns).
    """
    print(f"Input: {adata.n_obs} cells x {adata.n_vars} genes")
    
    # 1. Make names unique
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    
    # 2. Basic filtering
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    print(f"After basic filtering: {adata.n_obs} cells x {adata.n_vars} genes")
    
    # 3. Mitochondrial QC
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print(f"Mitochondrial gene stats:")
    print(f"  Cells with >0% MT: {(adata.obs['pct_counts_mt'] > 0).sum()}")
    print(f"  Mean MT%: {adata.obs['pct_counts_mt'].mean():.2f}")
    print(f"  Max MT%: {adata.obs['pct_counts_mt'].max():.2f}")
    
    # Plot QC metrics
    sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
                 jitter=0.4, multi_panel=True)
    
    # Filter by MT%
    n_before = adata.n_obs
    adata = adata[adata.obs['pct_counts_mt'] < mt_pct_threshold, :].copy()
    print(f"Filtered MT% > {mt_pct_threshold}: removed {n_before - adata.n_obs} cells")
    print(f"After MT filtering: {adata.n_obs} cells x {adata.n_vars} genes")
    
    # 4. Normalize and log-transform
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    
    # 5. Store raw counts for differential expression
    adata.raw = adata
    
    # 6. HVG selection
    sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes)
    print(f"Highly variable genes: {adata.var['highly_variable'].sum()}")
    adata = adata[:, adata.var.highly_variable].copy()
    
    # 7. Scale, PCA, neighbors, UMAP, leiden
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata)
    sc.pp.neighbors(adata, n_pcs=n_pcs)
    sc.tl.umap(adata)
    sc.tl.leiden(adata, resolution=leiden_resolution)
    
    print(f"Output: {adata.n_obs} cells x {adata.n_vars} genes")
    print(f"Leiden clusters: {adata.obs['leiden'].nunique()}")
    
    return adata

print("Preprocessing function defined.")

In [ ]:
# Cell 3: cellxGene Metadata Annotation Function
#
# Adds required fields for cellxGene schema compliance.
# See: https://github.com/chanzuckerberg/single-cell-curation/blob/main/schema/5.0.0/schema.md

def annotate_cellxgene_metadata(adata, title, assay_ontology_term_id="EFO:0009922",
                                 disease_ontology_term_id="unknown",
                                 tissue_ontology_term_id="UBERON:0001264",
                                 organism_ontology_term_id="NCBITaxon:9606"):
    """Add required cellxGene schema fields to an AnnData object.
    
    Args:
        adata: AnnData to annotate.
        title: Dataset title for uns.
        assay_ontology_term_id: EFO term for assay (default: 10x 3' v2).
        disease_ontology_term_id: MONDO term for disease.
        tissue_ontology_term_id: UBERON term for tissue (default: pancreas).
        organism_ontology_term_id: NCBITaxon term (default: human).
    """
    # obs fields
    adata.obs['organism_ontology_term_id'] = organism_ontology_term_id
    adata.obs['tissue_ontology_term_id'] = tissue_ontology_term_id
    adata.obs['assay_ontology_term_id'] = assay_ontology_term_id
    adata.obs['disease_ontology_term_id'] = disease_ontology_term_id
    
    if 'cell_type_ontology_term_id' not in adata.obs.columns:
        adata.obs['cell_type_ontology_term_id'] = 'unknown'
    if 'donor_id' not in adata.obs.columns:
        adata.obs['donor_id'] = 'unknown'
    if 'suspension_type' not in adata.obs.columns:
        adata.obs['suspension_type'] = 'cell'
    if 'is_primary_data' not in adata.obs.columns:
        adata.obs['is_primary_data'] = True
    if 'sex_ontology_term_id' not in adata.obs.columns:
        adata.obs['sex_ontology_term_id'] = 'unknown'
    if 'development_stage_ontology_term_id' not in adata.obs.columns:
        adata.obs['development_stage_ontology_term_id'] = 'unknown'
    if 'self_reported_ethnicity_ontology_term_id' not in adata.obs.columns:
        adata.obs['self_reported_ethnicity_ontology_term_id'] = 'unknown'
    
    # var fields
    if 'feature_is_filtered' not in adata.var.columns:
        adata.var['feature_is_filtered'] = False
    if 'feature_name' not in adata.var.columns:
        adata.var['feature_name'] = adata.var_names
    if 'feature_biotype' not in adata.var.columns:
        adata.var['feature_biotype'] = 'gene'
    
    # uns fields
    adata.uns['schema_version'] = '5.0.0'
    adata.uns['title'] = title
    if 'X_umap' in adata.obsm:
        adata.uns['default_embedding'] = 'X_umap'
    
    print(f"cellxGene metadata annotated for: {title}")
    return adata

print("cellxGene annotation function defined.")

In [ ]:
# Cell 4: Validation Function
#
# Checks that all required cellxGene fields are present before export.

def validate_cellxgene(adata, name="dataset"):
    """Validate that required cellxGene schema fields are present."""
    issues = []
    
    # Required obs fields
    required_obs = [
        'organism_ontology_term_id', 'tissue_ontology_term_id',
        'assay_ontology_term_id', 'disease_ontology_term_id',
        'cell_type_ontology_term_id', 'donor_id', 'suspension_type',
        'is_primary_data', 'sex_ontology_term_id',
        'development_stage_ontology_term_id',
        'self_reported_ethnicity_ontology_term_id',
    ]
    for col in required_obs:
        if col not in adata.obs.columns:
            issues.append(f"Missing obs column: {col}")
    
    # Required var fields
    required_var = ['feature_is_filtered', 'feature_name', 'feature_biotype']
    for col in required_var:
        if col not in adata.var.columns:
            issues.append(f"Missing var column: {col}")
    
    # Required uns fields
    required_uns = ['schema_version', 'title']
    for key in required_uns:
        if key not in adata.uns:
            issues.append(f"Missing uns key: {key}")
    
    # Check embeddings
    if 'X_umap' not in adata.obsm:
        issues.append("Missing obsm embedding: X_umap")
    
    if issues:
        print(f"VALIDATION FAILED for {name}:")
        for issue in issues:
            print(f"  - {issue}")
        return False
    else:
        print(f"VALIDATION PASSED for {name}")
        print(f"  {adata.n_obs} cells x {adata.n_vars} genes")
        print(f"  Title: {adata.uns['title']}")
        return True

print("Validation function defined.")

---
## GSE154778: Primary vs Metastatic PDAC

**Description:** 10 primary tumors + 6 metastatic PDAC samples  
**Files:** `GSE154778_dgeMtx.csv.gz` (DGE matrix), `GSE154778_RAW.tar` (per-sample 10x files)  
**Strategy:** Try the CSV DGE matrix first. If it lacks per-sample metadata, fall back to the TAR.

In [ ]:
# Cell 5: GSE154778 — Load CSV DGE matrix

csv_path = os.path.join(DRIVE_BASE, "GSE154778", "GSE154778_dgeMtx.csv.gz")

if os.path.exists(csv_path):
    print(f"Loading {csv_path}...")
    # DGE matrices are typically genes (rows) x cells (columns)
    dge = pd.read_csv(csv_path, index_col=0)
    print(f"Raw DGE shape: {dge.shape}")
    print(f"First 5 row names (likely genes): {dge.index[:5].tolist()}")
    print(f"First 5 column names (likely cells): {dge.columns[:5].tolist()}")
    print(f"\nData preview:")
    display(dge.iloc[:5, :5])
else:
    print(f"CSV not found at {csv_path}")
    print("Will use TAR file instead (see cells below).")
    dge = None

In [ ]:
# Cell 6: GSE154778 — Inspect and create AnnData from CSV

if dge is not None:
    # Transpose: genes x cells -> cells x genes (AnnData convention)
    adata_154778 = ad.AnnData(X=sp.csr_matrix(dge.values.T), 
                              obs=pd.DataFrame(index=dge.columns),
                              var=pd.DataFrame(index=dge.index))
    print(f"AnnData created: {adata_154778.shape}")
    print(f"\nGene name examples: {adata_154778.var_names[:10].tolist()}")
    print(f"Cell barcode examples: {adata_154778.obs_names[:5].tolist()}")
    
    # Try to extract sample info from barcode names
    # Common patterns: SAMPLE_BARCODE or BARCODE-SAMPLE
    barcodes = adata_154778.obs_names.tolist()
    print(f"\nBarcode patterns (first 10):")
    for b in barcodes[:10]:
        print(f"  {b}")
else:
    print("No CSV DGE matrix loaded. Proceeding to TAR extraction.")

In [ ]:
# Cell 7: GSE154778 — Extract and inspect TAR (fallback or for sample metadata)

tar_path = os.path.join(DRIVE_BASE, "GSE154778", "GSE154778_RAW.tar")
extract_dir = os.path.join(DRIVE_BASE, "GSE154778", "GSE154778_RAW")

if os.path.exists(tar_path):
    if not os.path.exists(extract_dir):
        print(f"Extracting {tar_path}...")
        with tarfile.open(tar_path, 'r') as tar:
            tar.extractall(path=extract_dir)
        print("Extraction complete.")
    else:
        print(f"Already extracted at {extract_dir}")
    
    # Inspect directory structure
    print(f"\nContents of {extract_dir}:")
    for root, dirs, files in os.walk(extract_dir):
        level = root.replace(extract_dir, '').count(os.sep)
        indent = '  ' * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = '  ' * (level + 1)
        for f in sorted(files)[:20]:  # limit output
            size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
            print(f"{subindent}{f} ({size_mb:.2f} MB)")
        if len(files) > 20:
            print(f"{subindent}... and {len(files) - 20} more files")
else:
    print(f"TAR file not found at {tar_path}")
    print("(GSE154778 CSV matrix was loaded successfully — TAR is not required.)")

In [ ]:
# Cell 8: GSE154778 — Load per-sample 10x files from TAR (if needed)
#
# After inspecting the TAR contents above, adjust the loading strategy.
# This cell handles the common case of per-sample directories with
# barcodes.tsv.gz, features.tsv.gz/genes.tsv.gz, matrix.mtx.gz.

# If we already have adata_154778 from the CSV with sufficient metadata, skip this.
# Otherwise, load from TAR-extracted per-sample 10x directories.

if 'adata_154778' not in dir() or adata_154778 is None:
    print("Loading from extracted TAR files...")
    
    # Find sample directories containing matrix.mtx.gz
    sample_dirs = []
    for root, dirs, files in os.walk(extract_dir):
        mtx_files = [f for f in files if f.endswith('matrix.mtx.gz') or f.endswith('matrix.mtx')]
        if mtx_files:
            sample_dirs.append(root)
    
    if sample_dirs:
        print(f"Found {len(sample_dirs)} sample directories with 10x matrices")
        
        adatas = []
        for sd in sorted(sample_dirs):
            sample_name = os.path.basename(sd)
            print(f"  Loading {sample_name}...")
            try:
                a = sc.read_10x_mtx(sd, var_names='gene_symbols', make_unique=True)
                a.obs['sample'] = sample_name
                adatas.append(a)
                print(f"    {a.n_obs} cells x {a.n_vars} genes")
            except Exception as e:
                print(f"    Failed: {e}")
        
        if adatas:
            adata_154778 = ad.concat(adatas, label='sample_key', keys=[a.obs['sample'].iloc[0] for a in adatas])
            print(f"\nConcatenated: {adata_154778.shape}")
    else:
        # Fallback: look for individual gz files (GSM-level)
        print("No standard 10x directories found. Inspecting for GSM-level files...")
        gz_files = glob.glob(os.path.join(extract_dir, '**', '*.gz'), recursive=True)
        print(f"Found {len(gz_files)} .gz files:")
        for f in sorted(gz_files)[:20]:
            print(f"  {os.path.relpath(f, extract_dir)}")
        print("\n** Manual loading required — adjust code based on file structure above **")
else:
    print(f"Using CSV-derived AnnData: {adata_154778.shape}")

In [ ]:
# Cell 9: GSE154778 — Add sample metadata
#
# Map samples to primary/metastatic condition and add donor IDs.
# Adjust the mapping based on the actual sample names found above.

print(f"AnnData shape: {adata_154778.shape}")

# Check what sample info we have
if 'sample' in adata_154778.obs.columns:
    print(f"\nSamples found:")
    print(adata_154778.obs['sample'].value_counts())
elif 'sample_key' in adata_154778.obs.columns:
    print(f"\nSample keys found:")
    print(adata_154778.obs['sample_key'].value_counts())
    adata_154778.obs['sample'] = adata_154778.obs['sample_key']

# Try to infer sample identity from barcodes if no sample column
if 'sample' not in adata_154778.obs.columns:
    print("\nNo sample column found. Inspecting barcodes for sample patterns...")
    barcodes = adata_154778.obs_names[:20].tolist()
    for b in barcodes:
        print(f"  {b}")
    print("\n** You may need to manually parse barcodes to extract sample IDs **")

# PLACEHOLDER: Map samples to conditions
# Uncomment and adjust after inspecting sample names above
# primary_samples = ['sample1', 'sample2', ...]  # 10 primary samples
# metastatic_samples = ['sample3', 'sample4', ...]  # 6 metastatic samples
# adata_154778.obs['condition'] = adata_154778.obs['sample'].map(
#     {**{s: 'primary' for s in primary_samples},
#      **{s: 'metastatic' for s in metastatic_samples}}
# )

# Add donor_id (same as sample for single-patient-per-sample studies)
if 'sample' in adata_154778.obs.columns:
    adata_154778.obs['donor_id'] = adata_154778.obs['sample']

print("\nobs columns:", adata_154778.obs.columns.tolist())

In [ ]:
# Cell 10: GSE154778 — QC, Preprocess, Visualize, and Save

# Run preprocessing
adata_154778 = preprocess_adata(adata_154778)

# UMAP plots
print("\n--- UMAP Visualizations ---")
sc.pl.umap(adata_154778, color='leiden', title='GSE154778 — Leiden Clusters')

if 'condition' in adata_154778.obs.columns:
    sc.pl.umap(adata_154778, color='condition', title='GSE154778 — Primary vs Metastatic')

if 'sample' in adata_154778.obs.columns:
    sc.pl.umap(adata_154778, color='sample', title='GSE154778 — Samples')

# Marker gene dotplot for cell type annotation
# Common pancreatic markers
pancreatic_markers = ['KRT19', 'EPCAM', 'CDH1',   # Epithelial/ductal
                      'ACTA2', 'COL1A1', 'LUM',    # Fibroblast/stellate
                      'PECAM1', 'VWF',              # Endothelial
                      'PTPRC', 'CD3D', 'CD68',      # Immune
                      'INS', 'GCG', 'SST',          # Endocrine
                      'PRSS1', 'AMY2A']             # Acinar
available_markers = [g for g in pancreatic_markers if g in adata_154778.var_names]
if available_markers:
    sc.pl.dotplot(adata_154778, var_names=available_markers, groupby='leiden',
                  title='GSE154778 — Pancreatic Marker Genes')

# Annotate cellxGene metadata
adata_154778 = annotate_cellxgene_metadata(
    adata_154778,
    title="GSE154778: Primary vs Metastatic PDAC scRNA-seq",
    disease_ontology_term_id="MONDO:0006047",  # PDAC
)

# Save
out_path = os.path.join(OUTPUT_DIR, "GSE154778_processed.h5ad")
adata_154778.write_h5ad(out_path)
print(f"\nSaved to {out_path}")
print(f"Final shape: {adata_154778.shape}")

---
## GSE162708: Metastatic Pancreatic Neuroendocrine Tumor

**Description:** 24,544 cells from 5 samples (2 primary tumors, liver metastasis, normal liver, PBMCs)  
**Files:** `GSE162708_barcodes.tsv.gz`, `GSE162708_features.tsv.gz`, `GSE162708_matrix.mtx.gz`  
**Strategy:** Standard 10x format — set up directory with correct filenames, then `sc.read_10x_mtx()`

In [ ]:
# Cell 11: GSE162708 — Set up 10x directory and load
#
# scanpy's read_10x_mtx expects standard filenames (barcodes.tsv.gz, features.tsv.gz,
# matrix.mtx.gz). We symlink the GSE-prefixed files from Drive into a temp directory.

import shutil

gse162708_src = os.path.join(DRIVE_BASE, "GSE162708")
tmpdir = tempfile.mkdtemp(prefix="GSE162708_10x_")

# Map GSE-prefixed files to standard 10x names
file_mapping = {
    "GSE162708_barcodes.tsv.gz": "barcodes.tsv.gz",
    "GSE162708_features.tsv.gz": "features.tsv.gz",
    "GSE162708_matrix.mtx.gz": "matrix.mtx.gz",
}

for src_name, dst_name in file_mapping.items():
    src = os.path.join(gse162708_src, src_name)
    dst = os.path.join(tmpdir, dst_name)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f"Linked {src_name} -> {dst_name}")
    else:
        print(f"WARNING: Source file not found: {src}")

# Load with scanpy
print("\nLoading 10x matrix...")
adata_162708 = sc.read_10x_mtx(tmpdir, var_names='gene_symbols', make_unique=True)
print(f"Loaded: {adata_162708.shape}")
print(f"\nGene examples: {adata_162708.var_names[:10].tolist()}")
print(f"Barcode examples: {adata_162708.obs_names[:5].tolist()}")

# Clean up temp directory
shutil.rmtree(tmpdir)
print(f"\nCleaned up temp directory.")

In [ ]:
# Cell 12: GSE162708 — Parse barcodes for sample identity
#
# 10x barcodes often end with -1, -2, etc. to indicate sample/GEM well.
# The 5 samples are: 2 primary tumors, liver metastasis, normal liver, PBMCs.

barcodes = adata_162708.obs_names.tolist()

# Check barcode suffixes
suffixes = [b.split('-')[-1] if '-' in b else 'none' for b in barcodes]
suffix_counts = pd.Series(suffixes).value_counts().sort_index()
print("Barcode suffix distribution:")
print(suffix_counts)

print(f"\nSample barcode examples (first 3 per suffix):")
for suffix in suffix_counts.index[:10]:
    examples = [b for b in barcodes if b.endswith(f'-{suffix}')][:3]
    print(f"  Suffix {suffix}: {examples}")

# Map suffixes to sample identities
# PLACEHOLDER: adjust after inspecting suffix distribution above
# Expected 5 samples: primary_tumor_1, primary_tumor_2, liver_met, normal_liver, pbmc
suffix_to_sample = {}
# suffix_to_sample = {
#     '1': 'primary_tumor_1',
#     '2': 'primary_tumor_2',
#     '3': 'liver_metastasis',
#     '4': 'normal_liver',
#     '5': 'pbmc',
# }

if suffix_to_sample:
    adata_162708.obs['sample'] = [suffix_to_sample.get(b.split('-')[-1], 'unknown') for b in barcodes]
    print("\nSample assignments:")
    print(adata_162708.obs['sample'].value_counts())
else:
    print("\n** Fill in suffix_to_sample mapping above based on suffix distribution **")
    # Temporary: use suffix as sample ID
    adata_162708.obs['sample'] = [b.split('-')[-1] if '-' in b else 'unknown' for b in barcodes]
    print("\nUsing barcode suffix as temporary sample ID:")
    print(adata_162708.obs['sample'].value_counts())

In [ ]:
# Cell 13: GSE162708 — Add per-sample metadata
#
# Set tissue ontology IDs per sample since tissues differ (pancreas, liver, blood).

# PLACEHOLDER: Adjust sample-to-condition and tissue mappings
# after confirming sample identities in the cell above.

# Tissue ontology mapping per sample type
tissue_map = {
    'primary_tumor_1': 'UBERON:0001264',   # pancreas
    'primary_tumor_2': 'UBERON:0001264',   # pancreas
    'liver_metastasis': 'UBERON:0002107',  # liver
    'normal_liver': 'UBERON:0002107',      # liver
    'pbmc': 'UBERON:0000178',              # blood
}

# Condition mapping
condition_map = {
    'primary_tumor_1': 'primary_tumor',
    'primary_tumor_2': 'primary_tumor',
    'liver_metastasis': 'metastasis',
    'normal_liver': 'normal',
    'pbmc': 'normal',
}

if 'sample' in adata_162708.obs.columns:
    samples = adata_162708.obs['sample'].unique()
    print(f"Unique samples: {samples}")
    
    # Apply tissue mapping if sample names match
    if all(s in tissue_map for s in samples):
        adata_162708.obs['tissue_ontology_term_id'] = adata_162708.obs['sample'].map(tissue_map)
        adata_162708.obs['condition'] = adata_162708.obs['sample'].map(condition_map)
        print("\nTissue and condition annotations applied.")
    else:
        print("\n** Sample names don't match expected mapping. Adjust tissue_map above. **")
        print(f"Expected: {list(tissue_map.keys())}")
        print(f"Got: {samples.tolist()}")

adata_162708.obs['donor_id'] = adata_162708.obs['sample']
print("\nobs columns:", adata_162708.obs.columns.tolist())

In [ ]:
# Cell 14: GSE162708 — QC, Preprocess, Visualize, and Save

# Run preprocessing
adata_162708 = preprocess_adata(adata_162708)

# UMAP plots
print("\n--- UMAP Visualizations ---")
sc.pl.umap(adata_162708, color='leiden', title='GSE162708 — Leiden Clusters')
sc.pl.umap(adata_162708, color='sample', title='GSE162708 — Samples')

if 'condition' in adata_162708.obs.columns:
    sc.pl.umap(adata_162708, color='condition', title='GSE162708 — Condition')

# Marker genes
net_markers = ['CHGA', 'SYP', 'NCAM1',     # Neuroendocrine
               'KRT19', 'EPCAM',             # Epithelial
               'PTPRC', 'CD3D', 'CD68',      # Immune
               'PECAM1', 'VWF',              # Endothelial
               'ALB', 'APOB',                # Hepatocytes
               'COL1A1', 'ACTA2']            # Fibroblast/stellate
available_markers = [g for g in net_markers if g in adata_162708.var_names]
if available_markers:
    sc.pl.dotplot(adata_162708, var_names=available_markers, groupby='leiden',
                  title='GSE162708 — Marker Genes')

# Annotate cellxGene metadata
adata_162708 = annotate_cellxgene_metadata(
    adata_162708,
    title="GSE162708: Metastatic Pancreatic Neuroendocrine Tumor scRNA-seq",
    disease_ontology_term_id="MONDO:0006130",  # Pancreatic neuroendocrine tumor
)

# Save
out_path = os.path.join(OUTPUT_DIR, "GSE162708_processed.h5ad")
adata_162708.write_h5ad(out_path)
print(f"\nSaved to {out_path}")
print(f"Final shape: {adata_162708.shape}")

---
## GSE165399: Normal Pancreas, IPMN, Adenosquamous Carcinoma

**Description:** 3 conditions — normal pancreas, IPMN, adenosquamous carcinoma (PASC)  
**Files:** `GSE165399_RAW.tar` (TAR of TXT files — likely tab-separated DGE matrices)  
**Strategy:** Extract TAR, inspect TXT file structure, load per-sample, concatenate

In [ ]:
# Cell 15: GSE165399 — Locate extracted files (or extract TAR)

extract_dir_165399 = os.path.join(DRIVE_BASE, "GSE165399", "GSE165399_RAW")
tar_path_165399 = os.path.join(DRIVE_BASE, "GSE165399", "GSE165399_RAW.tar")

if os.path.isdir(extract_dir_165399):
    print(f"Pre-extracted directory found at {extract_dir_165399}")
elif os.path.exists(tar_path_165399):
    print(f"Extracting {tar_path_165399}...")
    with tarfile.open(tar_path_165399, 'r') as tar:
        tar.extractall(path=extract_dir_165399)
    print("Extraction complete.")
else:
    print(f"ERROR: Neither extracted dir nor TAR found on Drive.")
    print(f"  Expected dir: {extract_dir_165399}")
    print(f"  Expected tar: {tar_path_165399}")

if os.path.isdir(extract_dir_165399):
    # List all extracted files
    print(f"\nContents of {extract_dir_165399}:")
    all_files = []
    for root, dirs, files in os.walk(extract_dir_165399):
        for f in sorted(files):
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1e6
            relpath = os.path.relpath(fpath, extract_dir_165399)
            print(f"  {relpath} ({size_mb:.2f} MB)")
            all_files.append(fpath)
    
    # Preview the first TXT file
    txt_files = [f for f in all_files if f.endswith('.txt') or f.endswith('.txt.gz')]
    if txt_files:
        print(f"\nPreview of first TXT file ({os.path.basename(txt_files[0])}):")
        try:
            preview = pd.read_csv(txt_files[0], sep='\t', nrows=5, index_col=0)
            print(f"  Shape (first 5 rows): {preview.shape}")
            print(f"  Row names: {preview.index.tolist()}")
            print(f"  Col names (first 5): {preview.columns[:5].tolist()}")
            display(preview.iloc[:5, :5])
        except Exception as e:
            print(f"  Could not parse as TSV: {e}")
            print("  Trying comma-separated...")
            preview = pd.read_csv(txt_files[0], sep=',', nrows=5, index_col=0)
            print(f"  Shape: {preview.shape}")
            display(preview.iloc[:5, :5])

In [ ]:
# Cell 16: GSE165399 — Load TXT files and create per-sample AnnData objects
#
# Each TXT file is expected to be a DGE matrix (genes x cells, tab-separated).
# Adjust sep and transpose based on the preview above.

txt_files = sorted(glob.glob(os.path.join(extract_dir_165399, '**', '*.txt*'), recursive=True))
print(f"Found {len(txt_files)} TXT files")

adatas_165399 = []
for fpath in txt_files:
    fname = os.path.basename(fpath)
    # Extract GSM accession from filename (e.g., GSM5032701_sample.txt.gz)
    gsm_id = fname.split('_')[0] if fname.startswith('GSM') else fname.split('.')[0]
    
    print(f"\nLoading {fname} (sample: {gsm_id})...")
    try:
        # Try tab-separated first
        df = pd.read_csv(fpath, sep='\t', index_col=0)
    except Exception:
        df = pd.read_csv(fpath, sep=',', index_col=0)
    
    print(f"  Raw shape: {df.shape}")
    print(f"  Row names (first 3): {df.index[:3].tolist()}")
    print(f"  Col names (first 3): {df.columns[:3].tolist()}")
    
    # DGE format is typically genes (rows) x cells (columns) — transpose for AnnData
    # If rows look like barcodes (ATCG...), data may already be cells x genes
    first_row_name = str(df.index[0])
    if all(c in 'ACGTN-0123456789' for c in first_row_name.replace('_', '')):
        # Rows are barcodes — already cells x genes
        print("  Format: cells x genes (no transpose needed)")
        adata_sample = ad.AnnData(X=sp.csr_matrix(df.values),
                                  obs=pd.DataFrame(index=df.index),
                                  var=pd.DataFrame(index=df.columns))
    else:
        # Rows are genes — transpose
        print("  Format: genes x cells (transposing)")
        adata_sample = ad.AnnData(X=sp.csr_matrix(df.values.T),
                                  obs=pd.DataFrame(index=df.columns),
                                  var=pd.DataFrame(index=df.index))
    
    adata_sample.obs['sample'] = gsm_id
    adatas_165399.append(adata_sample)
    print(f"  AnnData: {adata_sample.shape}")

if adatas_165399:
    adata_165399 = ad.concat(adatas_165399, label='sample_key',
                             keys=[a.obs['sample'].iloc[0] for a in adatas_165399])
    print(f"\nConcatenated AnnData: {adata_165399.shape}")
    print(f"Samples: {adata_165399.obs['sample'].value_counts()}")

In [ ]:
# Cell 17: GSE165399 — Add metadata (condition mapping)
#
# Map GSM accessions to conditions: normal, IPMN, PASC (adenosquamous carcinoma)
# Adjust the mapping after inspecting the actual GSM IDs above.

print("Samples in dataset:")
print(adata_165399.obs['sample'].value_counts())

# PLACEHOLDER: Map GSM accessions to conditions
# Check GEO page for GSE165399 to get the correct mapping
gsm_to_condition = {}
# Example (fill in after checking GEO):
# gsm_to_condition = {
#     'GSM5032701': 'normal',
#     'GSM5032702': 'IPMN',
#     'GSM5032703': 'PASC',
# }

if gsm_to_condition:
    adata_165399.obs['condition'] = adata_165399.obs['sample'].map(gsm_to_condition)
    print("\nCondition mapping applied:")
    print(adata_165399.obs['condition'].value_counts())
else:
    print("\n** Fill in gsm_to_condition mapping above using GEO metadata **")
    print("Check: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE165399")

# Disease ontology mapping
condition_to_disease = {
    'normal': 'PATO:0000461',       # normal
    'IPMN': 'MONDO:0006047',        # intraductal papillary mucinous neoplasm
    'PASC': 'MONDO:0003450',        # adenosquamous carcinoma
}

adata_165399.obs['donor_id'] = adata_165399.obs['sample']
print("\nobs columns:", adata_165399.obs.columns.tolist())

In [ ]:
# Cell 18: GSE165399 — QC, Preprocess, Visualize, and Save

# Run preprocessing
adata_165399 = preprocess_adata(adata_165399)

# UMAP plots
print("\n--- UMAP Visualizations ---")
sc.pl.umap(adata_165399, color='leiden', title='GSE165399 — Leiden Clusters')
sc.pl.umap(adata_165399, color='sample', title='GSE165399 — Samples')

if 'condition' in adata_165399.obs.columns:
    sc.pl.umap(adata_165399, color='condition', title='GSE165399 — Condition')

# Marker genes
markers_165399 = ['KRT19', 'EPCAM', 'CDH1',   # Epithelial/ductal
                  'MUC1', 'MUC2', 'MUC5AC',    # Mucin (IPMN markers)
                  'TP63', 'KRT5', 'KRT14',      # Squamous (PASC markers)
                  'PTPRC', 'CD3D', 'CD68',      # Immune
                  'PECAM1', 'VWF',              # Endothelial
                  'COL1A1', 'ACTA2',            # Fibroblast
                  'INS', 'GCG', 'SST',          # Endocrine
                  'PRSS1', 'AMY2A']             # Acinar
available_markers = [g for g in markers_165399 if g in adata_165399.var_names]
if available_markers:
    sc.pl.dotplot(adata_165399, var_names=available_markers, groupby='leiden',
                  title='GSE165399 — Marker Genes')

# Annotate cellxGene metadata
adata_165399 = annotate_cellxgene_metadata(
    adata_165399,
    title="GSE165399: Normal Pancreas, IPMN, and Adenosquamous Carcinoma scRNA-seq",
    disease_ontology_term_id="unknown",  # varies per sample — set per-cell below
)

# Set per-cell disease ontology if condition mapping exists
if 'condition' in adata_165399.obs.columns and adata_165399.obs['condition'].notna().all():
    adata_165399.obs['disease_ontology_term_id'] = adata_165399.obs['condition'].map(
        condition_to_disease
    ).fillna('unknown')

# Save
out_path = os.path.join(OUTPUT_DIR, "GSE165399_processed.h5ad")
adata_165399.write_h5ad(out_path)
print(f"\nSaved to {out_path}")
print(f"Final shape: {adata_165399.shape}")

---
## Final Validation & Summary

In [ ]:
# Cell 19: Validate all three .h5ad files

datasets = {
    "GSE154778": os.path.join(OUTPUT_DIR, "GSE154778_processed.h5ad"),
    "GSE162708": os.path.join(OUTPUT_DIR, "GSE162708_processed.h5ad"),
    "GSE165399": os.path.join(OUTPUT_DIR, "GSE165399_processed.h5ad"),
}

print("=" * 60)
print("VALIDATION RESULTS")
print("=" * 60)

for name, path in datasets.items():
    print(f"\n--- {name} ---")
    if os.path.exists(path):
        a = sc.read_h5ad(path)
        validate_cellxgene(a, name)
        
        # Check embeddings
        has_umap = 'X_umap' in a.obsm
        has_pca = 'X_pca' in a.obsm
        print(f"  Embeddings: UMAP={'yes' if has_umap else 'NO'}, PCA={'yes' if has_pca else 'NO'}")
        
        # Check conditions
        if 'condition' in a.obs.columns:
            print(f"  Conditions: {a.obs['condition'].value_counts().to_dict()}")
        if 'leiden' in a.obs.columns:
            print(f"  Leiden clusters: {a.obs['leiden'].nunique()}")
    else:
        print(f"  FILE NOT FOUND: {path}")

In [ ]:
# Cell 20: Summary table

summary_rows = []
for name, path in datasets.items():
    if os.path.exists(path):
        a = sc.read_h5ad(path)
        conditions = a.obs['condition'].nunique() if 'condition' in a.obs.columns else 'N/A'
        samples = a.obs['sample'].nunique() if 'sample' in a.obs.columns else 'N/A'
        size_mb = os.path.getsize(path) / 1e6
        summary_rows.append({
            'Dataset': name,
            'Cells': a.n_obs,
            'Genes': a.n_vars,
            'Samples': samples,
            'Conditions': conditions,
            'Clusters': a.obs['leiden'].nunique() if 'leiden' in a.obs.columns else 'N/A',
            'File Size (MB)': f"{size_mb:.1f}",
        })
    else:
        summary_rows.append({'Dataset': name, 'Cells': 'NOT FOUND'})

summary_df = pd.DataFrame(summary_rows)
print("\nDataset Summary:")
display(summary_df)

print("\nAll output files:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {f} ({size_mb:.1f} MB)")